# trial-matcher — judge ablation

Judges the same shortlist with several local models, to see which one reads
eligibility criteria best. Nothing here touches the index or the test set: the
shortlist is fixed, produced once by the retrieval stage.

**Before running:** Settings -> Accelerator: **GPU T4 x2**, Internet: **On**,
and attach the `trial-matcher-pack` dataset (built by `src.analysis.pack`).

Leave `SMOKE = True` for the first run: one trial, a couple of minutes, enough
to prove that the models load and the verdicts are written where they should be.

In [ ]:
SMOKE = True

REPO = "https://github.com/massimo-ruggiero/trial-matcher.git"
PACK = "/kaggle/input/trial-matcher-pack"
YEAR, RUN = 2021, "dense_medembed-small"

# Same family, same sizes: the only difference is the medical fine-tune.
MODELS = ["gemma3:4b", "medgemma:4b", "gemma3:27b", "medgemma:27b"]
TOPICS, DEPTH = "12-31", 20

if SMOKE:
    MODELS, TOPICS, DEPTH = ["gemma3:4b"], "12", 1

print(f"{len(MODELS)} models x topics {TOPICS} x top-{DEPTH}")

## 1. Ollama

In [ ]:
import os
import subprocess
import time

import requests

# Not /kaggle/working: the models are tens of gigabytes and everything there is
# saved as the notebook's output.
os.environ["OLLAMA_MODELS"] = "/tmp/ollama"

subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.Popen(
    ["ollama", "serve"],
    start_new_session=True,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        if requests.get("http://localhost:11434", timeout=2).ok:
            print("ollama is up")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError(open("/tmp/ollama.log").read())

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)

## 2. Code and dependencies

Only what judging needs: no encoder, no index, no torch.

In [ ]:
!git clone -q {REPO} /tmp/trial-matcher
!pip install -q typer tqdm python-dotenv pytrec-eval-terrier
!cd /tmp/trial-matcher && git log --oneline -1

## 3. Where things are read and written

The pack is mounted read-only, so the verdicts go to `/kaggle/working/out`.
If the pack carries a `verdicts` file from an earlier session it is brought
forward: the cache key is `(topic, trial, model, prompt)`, so whatever is
already judged is skipped instead of being paid for twice.

In [ ]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working")
os.environ["DATA_PROCESSED"] = f"{PACK}/processed"
os.environ["RUNS"] = str(WORK / "runs")
os.environ["DATA_OUT"] = str(WORK / "out")

(WORK / "runs").mkdir(exist_ok=True)
(WORK / "out").mkdir(exist_ok=True)
shutil.copy(f"{PACK}/runs/{RUN}{YEAR}.txt", WORK / "runs")

previous = Path(PACK) / f"verdicts{YEAR}.jsonl"
if previous.exists():
    shutil.copy(previous, WORK / "out")
    print(f"resuming from {sum(1 for _ in open(previous))} verdicts")

print(subprocess.run(["ls", "-R", PACK], capture_output=True, text=True).stdout)

## 4. Judge

One model at a time, using both GPUs. Each model is removed after its turn:
two 27B models are 34 GB and the disk does not hold them together.

In [ ]:
for model in MODELS:
    print(f"\n===== {model} =====", flush=True)
    subprocess.run(["ollama", "pull", model], check=True)

    cmd = ["python", "-m", "src.assess.judge", "--year", str(YEAR), "--run", RUN]
    cmd += ["--topics", TOPICS, "--depth", str(DEPTH), "--model", model]

    started = time.time()
    subprocess.run(cmd, cwd="/tmp/trial-matcher", check=True)
    print(f"{model}: {(time.time() - started) / 60:.1f} min")

    subprocess.run(["ollama", "rm", model], check=True)

## 5. What came out

In [ ]:
import json
from collections import Counter

path = WORK / "out" / f"verdicts{YEAR}.jsonl"
rows = [json.loads(line) for line in open(path)]
counts = Counter((r["model"], r["prompt_version"]) for r in rows)
for (model, prompt), n in sorted(counts.items()):
    seconds = [r.get("seconds", 0) for r in rows if r["model"] == model]
    mean = sum(seconds) / max(len(seconds), 1)
    print(f"{model:<16} prompt {prompt}  {n:>5} trials  {mean:5.1f} s each")

print(f"\n{path}  ({path.stat().st_size / 1e6:.1f} MB)")

## 6. Then

Save Version, download `out/verdicts2021.jsonl`, and append it to
`data/processed/verdicts2021.jsonl` in the repo. The rerank runs locally, needs
no model, and compares the judges from those records alone:

```bash
uv run python -m src.assess.rerank --run dense_medembed-small --depth 20 --model medgemma:4b
```

To carry on in a later session, upload that same file into a new version of the
pack dataset: cell 3 picks it up and only the missing trials are judged.